# Drone Swarm Attack Simulation Lab
**Senior Simulation Engineering & AI Systems**
This interactive environment demonstrates advanced swarm behavior modeling, probabilistic radar detection logic, and heuristic-based automated interception strategies addressing CLO-1 and CLO-2.

In [ ]:
# Cell 1 - Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.lines import Line2D
import math
import random
from IPython.display import HTML, display
import os

In [ ]:
# Cell 2 - Config & Parameters
class Config:
    WIDTH = 100
    HEIGHT = 100
    TARGET_POS = np.array([50.0, 50.0])
    TARGET_RADIUS = 5.0
    TARGET_HEALTH = 100
    RADAR_RANGE = 40.0
    RADAR_NOISE_PROB = 0.05  # Probability of target escaping ping
    INTERCEPT_RANGE = 20.0
    FRAMES = 300
    INTERVAL = 50
    DRONE_SPEED_BASE = 0.5
    MAX_DRONES = 15

In [ ]:
# Cell 3 - Simulation Environment & Drones
class Drone:
    def __init__(self, drone_id, start_pos):
        self.id = drone_id
        self.pos = np.array(start_pos, dtype=float)
        self.state = 'normal'
        self.speed = Config.DRONE_SPEED_BASE + random.uniform(-0.1, 0.3)
        
    def path_planning_bfs_heuristic(self, target_pos):
        direction = target_pos - self.pos
        dist = np.linalg.norm(direction)
        if dist == 0:
            return np.zeros(2)
        norm_dir = direction / dist
        evasion_noise = np.random.normal(0, 0.15, 2)
        return norm_dir + evasion_noise

    def move(self, target_pos):
        if self.state in ['intercepted', 'jammed', 'reached_target', 'damaged_target']:
            return
        direction = target_pos - self.pos
        dist = np.linalg.norm(direction)
        if dist < Config.TARGET_RADIUS:
            self.state = 'reached_target'
            return
        movement_vector = self.path_planning_bfs_heuristic(target_pos)
        self.pos += movement_vector * self.speed
        self.pos[0] = np.clip(self.pos[0], 0, Config.WIDTH)
        self.pos[1] = np.clip(self.pos[1], 0, Config.HEIGHT)

class Environment:
    def __init__(self):
        self.target_pos = Config.TARGET_POS
        self.target_health = Config.TARGET_HEALTH
    def check_target_damage(self, drones):
        damage = 0
        for d in drones:
            if d.state == 'reached_target':
                damage += 20
                d.state = 'damaged_target'
        self.target_health -= damage
        self.target_health = max(0, self.target_health)

In [ ]:
# Cell 4 - Detection & Tracking
class Radar:
    def __init__(self, pos, range_radius):
        self.pos = np.array(pos)
        self.range = range_radius
    def scan(self, drones):
        detected_ids = []
        for d in drones:
            if d.state in ['intercepted', 'jammed', 'reached_target', 'damaged_target']:
                continue
            dist = np.linalg.norm(d.pos - self.pos)
            if dist <= self.range:
                if random.random() > Config.RADAR_NOISE_PROB:
                    detected_ids.append(d.id)
                    if d.state == 'normal':
                        d.state = 'detected'
        return detected_ids

class Tracker:
    def __init__(self):
        self.tracks = {}
    def update(self, detected_ids, drones):
        active_tracks = []
        for d in drones:
            if d.id in detected_ids:
                if d.id not in self.tracks:
                    self.tracks[d.id] = {'first_seen': d.pos.copy(), 'history': []}
                self.tracks[d.id]['history'].append(d.pos.copy())
                active_tracks.append(d.id)
        return active_tracks

In [ ]:
# Cell 5 - Defense & Interception Strategies
class Interceptor:
    def __init__(self, mode='destroy'):
        self.mode = mode
    def neutralize(self, drone):
        if self.mode == 'destroy': drone.state = 'intercepted'
        elif self.mode == 'jam': drone.state = 'jammed'

class DefenseStrategy:
    def __init__(self, target_pos, intercept_range, interceptor):
        self.target_pos = target_pos
        self.intercept_range = intercept_range
        self.interceptor = interceptor
    def evaluate_and_intercept(self, tracked_ids, drones): pass

class ZoneDefense(DefenseStrategy):
    def evaluate_and_intercept(self, tracked_ids, drones):
        intercepted = []
        for d in drones:
            if d.id in tracked_ids and d.state == 'detected':
                dist = np.linalg.norm(d.pos - self.target_pos)
                if dist <= self.intercept_range:
                    self.interceptor.neutralize(d)
                    intercepted.append(d.id)
        return intercepted

class ActiveInterception(DefenseStrategy):
    def evaluate_and_intercept(self, tracked_ids, drones):
        intercepted = []
        threats = []
        for d in drones:
            if d.id in tracked_ids and d.state == 'detected':
                dist = np.linalg.norm(d.pos - self.target_pos)
                if dist <= self.intercept_range + 5:
                    threats.append((dist, d))
        threats.sort(key=lambda x: x[0])
        for i in range(min(3, len(threats))):
            target_drone = threats[i][1]
            self.interceptor.neutralize(target_drone)
            intercepted.append(target_drone.id)
        return intercepted

In [ ]:
# Cell 6 - Swarm & Scenario Engine
class SwarmEngine:
    def __init__(self, defense_strategy='active'):
        self.drones = []
        self.env = Environment()
        self.radar = Radar(self.env.target_pos, Config.RADAR_RANGE)
        self.tracker = Tracker()
        self.interceptor = Interceptor(mode='destroy')
        if defense_strategy == 'active':
            self.defense = ActiveInterception(self.env.target_pos, Config.INTERCEPT_RANGE, self.interceptor)
        else:
            self.defense = ZoneDefense(self.env.target_pos, Config.INTERCEPT_RANGE, self.interceptor)
        self.metrics = {'spawned': 0, 'detected': 0, 'intercepted': 0, 'reached_target': 0}
        
    def generate_scenario(self, count):
        for i in range(count):
            edge = random.choice(['top', 'bottom', 'left', 'right'])
            if edge == 'top': pos = [random.uniform(0, Config.WIDTH), Config.HEIGHT]
            elif edge == 'bottom': pos = [random.uniform(0, Config.WIDTH), 0]
            elif edge == 'left': pos = [0, random.uniform(0, Config.HEIGHT)]
            else: pos = [Config.WIDTH, random.uniform(0, Config.HEIGHT)]
            self.drones.append(Drone(i, pos))
            self.metrics['spawned'] += 1

    def step(self):
        for d in self.drones: d.move(self.env.target_pos)
        self.env.check_target_damage(self.drones)
        detected_ids = self.radar.scan(self.drones)
        tracked_ids = self.tracker.update(detected_ids, self.drones)
        if self.env.target_health > 0:
            self.defense.evaluate_and_intercept(tracked_ids, self.drones)
            
        self.metrics['detected'] = sum(1 for d in self.drones if d.state in ['detected', 'intercepted', 'jammed', 'reached_target', 'damaged_target'])
        self.metrics['intercepted'] = sum(1 for d in self.drones if d.state in ['intercepted', 'jammed'])
        self.metrics['reached_target'] = sum(1 for d in self.drones if d.state in ['reached_target', 'damaged_target'])
        return self.drones, self.env

In [ ]:
# Cell 7 - Visualization
def create_animation(engine, frames=Config.FRAMES):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(0, Config.WIDTH)
    ax.set_ylim(0, Config.HEIGHT)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.6, zorder=1)
    
    target_circle = plt.Circle(Config.TARGET_POS, Config.TARGET_RADIUS, color='green', alpha=0.8, zorder=2)
    ax.add_patch(target_circle)
    radar_circle = plt.Circle(Config.TARGET_POS, Config.RADAR_RANGE, color='blue', fill=False, linestyle='--', alpha=0.5, zorder=2)
    ax.add_patch(radar_circle)
    defense_circle = plt.Circle(Config.TARGET_POS, Config.INTERCEPT_RANGE, color='red', fill=False, linestyle='-.', alpha=0.5, zorder=2)
    ax.add_patch(defense_circle)
    
    scat = ax.scatter([], [], c=[], s=40, edgecolors='black', zorder=5)
    status_text = ax.text(0.02, 0.96, '', transform=ax.transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.5'), verticalalignment='top', zorder=10)
    
    state_colors = {'normal': '#1E90FF', 'detected': '#FFA500', 'intercepted': '#000000', 'jammed': '#555555', 'reached_target': '#FF0000', 'damaged_target': '#FF0000'}
    
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#1E90FF', markersize=8, label='Normal (Stealth)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFA500', markersize=8, label='Detected'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#000000', markersize=8, label='Intercepted'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF0000', markersize=8, label='Breached')
    ]
    ax.legend(handles=[target_circle, radar_circle, defense_circle] + legend_elements, labels=['Target', 'Radar Range', 'Defense Zone', 'Normal', 'Detected', 'Intercepted', 'Breached'], loc='upper right', fontsize=8)
    ax.set_title("Advanced Drone Swarm Defense Simulation", fontweight='bold')
    
    def update(frame):
        drones, env = engine.step()
        positions = [d.pos for d in drones]
        colors = [state_colors[d.state] for d in drones]
        
        if positions:
            scat.set_offsets(positions)
            scat.set_color(colors)
            
        health_pct = env.target_health / Config.TARGET_HEALTH
        target_circle.set_color((1.0 - health_pct, health_pct, 0.0))
        
        stats = f"⏱️ Frame: {frame}/{frames}\n"
        stats += f"🛡️ Target HP: {env.target_health}/{Config.TARGET_HEALTH}\n"
        stats += f"===================\n"
        stats += f"🛸 Swarm Spawned: {engine.metrics['spawned']}\n"
        stats += f"📡 Drones Detected: {engine.metrics['detected']}\n"
        stats += f"💥 Intercepted / Dead: {engine.metrics['intercepted']}\n"
        stats += f"🔥 Breaches Failed: {engine.metrics['reached_target']}"
        status_text.set_text(stats)
        
        if env.target_health <= 0:
            ax.set_title("❌ CRITICAL FAILURE: TARGET DESTROYED", color='red', fontweight='bold')
            
        return scat, target_circle, status_text
        
    ani = animation.FuncAnimation(fig, update, frames=frames, interval=Config.INTERVAL, blit=False, repeat=False)
    return fig, ani

In [ ]:
# Cell 8 - Run & Display Analytics
print("🚁 STARTING SWARM SIMULATION & ANALYSIS...")
engine = SwarmEngine(defense_strategy='active')
engine.generate_scenario(Config.MAX_DRONES)

fig, ani = create_animation(engine, frames=Config.FRAMES)
display(HTML(ani.to_jshtml()))
plt.close()

efficiency = 0
if engine.metrics['detected'] > 0:
    efficiency = (engine.metrics['intercepted'] / engine.metrics['detected']) * 100

print("\n" + "="*40)
print("📊 POST-SIMULATION CAPABILITY REPORT")
print("="*40)
print(f"Total Drones Invading: {engine.metrics['spawned']}")
print(f"Total Drones Tracked:  {engine.metrics['detected']}")
print(f"Total Neutralized:     {engine.metrics['intercepted']}")
print(f"Total Target Breaches: {engine.metrics['reached_target']}")
print(f"Target Final Health:   {engine.env.target_health} / {Config.TARGET_HEALTH}")
print(f"\nSystem Efficiency Rating: {efficiency:.1f}%")
print("="*40)